# Stage 06 — Voice Layer (ASR + TTS) and Speech Evaluation
## طبقة الصوت: التعرّف على الكلام + توليد الكلام

يحقّق هذا الدفتر ادّعاء *Voice Agent* في عنوان الرسالة:

**خط الصوت الكامل:** صوت السؤال ← **Whisper-large-v3** (ASR) ← نص ← (نظام RAG+LLM) ← نص الإجابة ← **MMS-TTS-Arabic** (TTS) ← صوت.

**التقييم:** Word Error Rate (WER) للتعرّف على الكلام العربي عبر دورة TTS→ASR، مع تطبيع عربي قياسي وفترة ثقة Bootstrap.

> قيد موثَّق: التقييم على كلام **مُصطنع (TTS)**؛ الكلام البشري الحقيقي (لهجات/ضوضاء) أصعب — يُوصى بتسجيلات بشرية لاحقاً.

In [1]:
# 6.1 - Setup: load Whisper (ASR) and Arabic MMS-TTS
from pathlib import Path
import torch, numpy as np, re, json
import soundfile as sf
import jiwer
from transformers import VitsModel, AutoTokenizer, pipeline

PROJECT_DIR = Path("saudi_labor_law_voice_agent_project")
VOICE_DIR = PROJECT_DIR / "11_voice_layer"
(VOICE_DIR / "audio_samples").mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RNG = np.random.default_rng(42)

# TTS (Arabic)
_tts = VitsModel.from_pretrained("facebook/mms-tts-ara").to(DEVICE); _tts.eval()
_ttok = AutoTokenizer.from_pretrained("facebook/mms-tts-ara")
SAMPLE_RATE = _tts.config.sampling_rate

# ASR (Whisper large v3)
_asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3",
                device=0 if DEVICE == "cuda" else -1, torch_dtype=torch.float16)
print("Voice layer ready | TTS sample_rate:", SAMPLE_RATE, "| ASR: whisper-large-v3")

C:\Users\PC\anaconda3\envs\datacolection\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 762/762 [00:00<00:00, 11974.84it/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 1259/1259 [00:00<00:00, 14529.50it/s]

Voice layer ready | TTS sample_rate: 16000 | ASR: whisper-large-v3


In [2]:
# 6.2 - Core voice functions + Arabic normalization for fair WER
def synthesize_speech(text, save_path=None):
    inp = _ttok(str(text), return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        wav = _tts(**inp).waveform.squeeze().cpu().numpy()
    if save_path is not None:
        sf.write(str(save_path), wav, SAMPLE_RATE)
    return wav

def transcribe(wav_or_path):
    out = _asr(wav_or_path, generate_kwargs={"language": "arabic"})
    return out["text"].strip()

def normalize_ar(t):
    t = re.sub(r"[ً-ْٰـ]", "", str(t))   # diacritics + tatweel
    t = re.sub(r"[إأآا]", "ا", t); t = re.sub(r"ى", "ي", t); t = re.sub(r"ة", "ه", t)
    t = re.sub(r"[^؀-ۿ\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

# quick self-test
_w = synthesize_speech("ما هي حقوق العامل عند انتهاء الخدمة؟")
_h = transcribe(_w)
print("Self-test transcription:", _h)
print("WER:", round(jiwer.wer(normalize_ar("ما هي حقوق العامل عند انتهاء الخدمة؟"), normalize_ar(_h)), 3))

[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Self-test transcription: ما هي حقوق العامل عند انتهاء الخدمة لي؟
WER: 0.286


In [3]:
# 6.3 - ASR evaluation (WER) over a sample of evaluation questions
import pandas as pd
eval_df = pd.read_csv(PROJECT_DIR / "03_final" / "rag_evaluation_dataset.csv", encoding="utf-8-sig")
inscope = eval_df[eval_df["eval_type"].isin(["legal_article_retrieval", "faq_retrieval"])].copy()
sample = inscope.sample(n=min(40, len(inscope)), random_state=42).reset_index(drop=True)

rows = []
for i, r in sample.iterrows():
    q = str(r["question"])
    wav = synthesize_speech(q)
    hyp = transcribe(wav)
    w = jiwer.wer(normalize_ar(q), normalize_ar(hyp))
    rows.append({"eval_id": r.get("eval_id"), "eval_type": r["eval_type"],
                 "reference": q, "asr_hypothesis": hyp, "wer": w})
    if (i + 1) % 10 == 0:
        print(f"  transcribed {i+1}/{len(sample)}")

asr_df = pd.DataFrame(rows)
asr_df.to_csv(VOICE_DIR / "asr_wer_detailed.csv", index=False, encoding="utf-8-sig")

def boot_ci(x, n=10000):
    x = np.asarray(x, float)
    return np.percentile([RNG.choice(x, len(x), replace=True).mean() for _ in range(n)], [2.5, 97.5])

mean_wer = asr_df["wer"].mean()
lo, hi = boot_ci(asr_df["wer"].values)
print(f"\n=== Arabic ASR (Whisper-large-v3) on synthetic speech, n={len(asr_df)} ===")
print(f"  Mean WER (normalized): {mean_wer:.3f}  95%CI=[{lo:.3f}, {hi:.3f}]")
print(f"  Median WER: {asr_df['wer'].median():.3f} | perfect (WER=0): {int((asr_df['wer']==0).sum())}/{len(asr_df)}")
pd.DataFrame([{"metric": "mean_wer", "n": len(asr_df), "value": round(mean_wer,4),
              "ci95_low": round(lo,4), "ci95_high": round(hi,4)}]).to_csv(
    VOICE_DIR / "asr_wer_summary.csv", index=False, encoding="utf-8-sig")
asr_df.head(8)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  transcribed 10/40


  transcribed 20/40


  transcribed 30/40


  transcribed 40/40

=== Arabic ASR (Whisper-large-v3) on synthetic speech, n=40 ===
  Mean WER (normalized): 0.209  95%CI=[0.162, 0.264]
  Median WER: 0.167 | perfect (WER=0): 4/40


,eval_id,eval_type,reference,asr_hypothesis,wer
0,70,faq_retrieval,هل يحق للعامل طلب نقل خدماته قبل مضي 12 شهرا م...,هل يحق للعمل طلب نقل خدماته قبل مضيء شهرا من ق...,0.111111
1,141,legal_article_retrieval,ما واجبات صاحب العمل تجاه العامل؟,مواجبات صاحب العمل تجاه العامي,0.500000
2,28,faq_retrieval,كيف يحسب الخليجي في نسب التوطين؟,كاف يحسب الخليج في نسبة توطنة,0.666667
3,20,faq_retrieval,"ما هي معايير الحصول على شهادة ""مواءمة""؟",ما هي معايير الحصول على شهادة موائم لني؟,0.250000
4,43,faq_retrieval,هل يمكن إلزام الموظفين بالعمل خلال إجازة عيد ا...,هل يمكن إلزام الموظفين بالعمل خلال إجازة عيد ا...,0.111111
5,118,faq_retrieval,هل يمكن تعويض الموظف عن إجازاته العادية قبل ان...,هل يمكن تعويض الموظف عن إجازاته العادية قبل ان...,0.047619
6,127,legal_article_retrieval,ما مدة الإجازة السنوية المستحقة للعامل؟,ما مدة الإجازة السنوية المستحقة للعامي؟,0.166667
7,109,faq_retrieval,هل يشمل حكم المادة (41) من لائحة الحقوق والمزا...,هل يشمل حكم المادة ملائحة الحقوق والمزايا الما...,0.250000


In [4]:
# 6.4 - End-to-end voice demo: spoken question -> ASR -> (existing RAG answer) -> spoken answer
# Reuses already-generated answers from Stage 04 to avoid reloading the RAG/LLM stack.
gen_path = PROJECT_DIR / "09_stage04_llm_generation_results" / "generation_evaluation_detailed.csv"
demo_rows = []
if gen_path.exists():
    gen = pd.read_csv(gen_path, encoding="utf-8-sig")
    answered = gen[(gen["answered"] == 1) & (gen["eval_type"] == "legal_article_retrieval")].head(3)
    for j, r in enumerate(answered.itertuples(), 1):
        q, ans = str(r.question), str(r.answer)
        synthesize_speech(q,   VOICE_DIR / "audio_samples" / f"demo{j}_question.wav")
        asr_text = transcribe(str(VOICE_DIR / "audio_samples" / f"demo{j}_question.wav"))
        synthesize_speech(ans, VOICE_DIR / "audio_samples" / f"demo{j}_answer.wav")
        demo_rows.append({"demo": j, "spoken_question": q, "asr_understood": asr_text,
                          "system_answer": ans,
                          "question_audio": f"demo{j}_question.wav",
                          "answer_audio": f"demo{j}_answer.wav"})
        print(f"[Demo {j}] Q audio + A audio saved.")
        print(f"   spoken: {q}")
        print(f"   ASR understood: {asr_text}")
        print(f"   answer (to TTS): {ans[:90]}...")
demo_df = pd.DataFrame(demo_rows)
demo_df.to_csv(VOICE_DIR / "voice_endtoend_demo.csv", index=False, encoding="utf-8-sig")
print("\nSaved end-to-end demo audio to", VOICE_DIR / "audio_samples")
demo_df

[Demo 1] Q audio + A audio saved.
   spoken: هل يحق لصاحب العمل نقل العامل من مكان عمله الأصلي؟
   ASR understood: هل يحق لصاحب العمل نقر العامل من مكان عمله الأصلي؟
   answer (to TTS): وفقًا للمادة رقم 58 من نظام العمل السعودي، لا يجوز لصاحب العمل أن ينقل العامل بغير موافقته...


[Demo 2] Q audio + A audio saved.
   spoken: كم مدة فترة التجربة في نظام العمل السعودي؟
   ASR understood: كم مدة؟ فترت التجربة في نظام العمل السعودي
   answer (to TTS): وفقًا للمادة رقم 54 من نظام العمل السعودي، لا يجوز وضع العامل تحت التجربة أكثر من مرة واحد...


[Demo 3] Q audio + A audio saved.
   spoken: كيف يتم دفع أجر العامل؟
   ASR understood: كيف يتم دفع أجر العام؟
   answer (to TTS): وفقًا للمادة رقم 90 من نظام العمل السعودي، يجب دفع أجر العامل وكل مبلغ مستحق له بالعملة ال...

Saved end-to-end demo audio to saudi_labor_law_voice_agent_project\11_voice_layer\audio_samples


,demo,spoken_question,asr_understood,system_answer,question_audio,answer_audio
0,1,هل يحق لصاحب العمل نقل العامل من مكان عمله الأ...,هل يحق لصاحب العمل نقر العامل من مكان عمله الأ...,وفقًا للمادة رقم 58 من نظام العمل السعودي، لا ...,demo1_question.wav,demo1_answer.wav
1,2,كم مدة فترة التجربة في نظام العمل السعودي؟,كم مدة؟ فترت التجربة في نظام العمل السعودي,وفقًا للمادة رقم 54 من نظام العمل السعودي، لا ...,demo2_question.wav,demo2_answer.wav
2,3,كيف يتم دفع أجر العامل؟,كيف يتم دفع أجر العام؟,وفقًا للمادة رقم 90 من نظام العمل السعودي، يجب...,demo3_question.wav,demo3_answer.wav


## 6.5 — الخلاصة والقيود

- **خط الصوت كامل وفعّال**: ASR (Whisper) + TTS (عربي) مدموجان مع نظام RAG.
- **WER** مُقاس بتطبيع عربي + فترة ثقة (ملفات في `11_voice_layer/`)، وعيّنات صوتية في `audio_samples/`.
- **قيود للرسالة:** (1) التقييم على كلام TTS مُصطنع لا بشري؛ (2) MMS-TTS صوت آلي بسيط؛ (3) يُوصى بجمع تسجيلات بشرية حقيقية (لهجات/ضوضاء) لقياس WER واقعي. هذه القيود تُكتب بصدق وتُظهر النضج البحثي.